# Học phân loại nhị phân với dữ liệu ung thư vú Wisconsin

Notebook này dạy từng bước: logistic regression, SVM, Naive Bayes, các chỉ số precision, recall, F1 và đường ROC. Trọng tâm là cách chọn chỉ số đánh giá khi hai loại lỗi có hậu quả khác nhau.

Dữ liệu: 569 khối u. Mỗi khối u có 30 đặc trưng tính từ ảnh chụp nhân tế bào (bán kính, độ nhám, chu vi, diện tích...). Mỗi đặc trưng có 3 phiên bản: mean (trung bình), error (sai số chuẩn) và worst (giá trị lớn nhất). Cột diagnosis: M là ác tính (malignant), B là lành tính (benign). Nguồn: Breast Cancer Wisconsin (Diagnostic), UCI, lấy từ scikit-learn.

Chuẩn bị:
1. Đặt file data5.csv cùng thư mục với notebook.
2. Cài thư viện nếu chưa có: pip install pandas numpy matplotlib seaborn scikit-learn

Cách làm: đọc phần giải thích, viết code vào ô trống, chạy bằng Shift + Enter, rồi so sánh với kết quả mong đợi. Nếu kết quả khác, đọc lại đề bài và thử lại trước khi xem đáp án.

## Bước 0: Import thư viện

Mục tiêu: nạp các thư viện dùng trong toàn bộ notebook. Ô này đã viết sẵn, bạn chỉ cần chạy.

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*penalty.*')  # sklearn 1.8 báo penalty sắp bị đổi tên, code vẫn chạy đúng
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, roc_curve, RocCurveDisplay,
                             PrecisionRecallDisplay, average_precision_score)

## Bước 1: Đọc dữ liệu

Mục tiêu: đưa file CSV vào DataFrame và kiểm tra kích thước.

Kiến thức: pd.read_csv đọc file thành bảng. df.shape trả về (số dòng, số cột).

Kết quả mong đợi: (569, 31).

In [ ]:
# Viết code ở đây:
# 1. Đọc file data5.csv vào biến df
# 2. In df.shape và tên 10 cột đầu


## Bước 2: Biến mục tiêu

Mục tiêu: đếm số khối u mỗi loại và tạo biến y dạng 0/1.

Kiến thức: trong y tế, lớp "dương tính" (positive, mã 1) thường là lớp bệnh, ở đây là M. Các chỉ số precision và recall đều tính theo lớp dương tính, nên chọn đúng lớp 1 rất quan trọng.

Kết quả mong đợi: B có 357 mẫu, M có 212 mẫu. Tỷ lệ ác tính khoảng 0,373. Không có ô trống.

In [ ]:
# Viết code ở đây:
# 1. In value_counts của diagnosis và tổng số ô trống
# 2. y = (df['diagnosis'] == 'M').astype(int)
# 3. X = df bỏ cột diagnosis
# 4. In tỷ lệ ác tính y.mean() (làm tròn 3 chữ số)


## Bước 3: So sánh u lành và u ác

Mục tiêu: xem đặc trưng nào khác biệt rõ giữa hai loại.

Kiến thức: boxplot cho trung vị, tứ phân vị và điểm ngoại lai của từng nhóm. Nếu hai hộp gần như không chồng nhau, đặc trưng đó phân biệt tốt.

Kết quả mong đợi: u ác có mean_radius trung bình khoảng 17,46, u lành khoảng 12,15. mean_concave_points khác nhau khoảng 3 lần (0,088 so với 0,026). mean_fractal_dimension gần như bằng nhau (0,063 ở cả hai nhóm).

Câu hỏi: đặc trưng nào gần như không giúp phân biệt hai loại?

In [ ]:
# Viết code ở đây:
# 1. Chọn 4 cột: mean_radius, mean_concave_points, mean_texture, mean_fractal_dimension
# 2. In trung bình của 4 cột này theo diagnosis (groupby, làm tròn 3 chữ số)
# 3. Vẽ 4 boxplot (sns.boxplot với x='diagnosis') trên plt.subplots(1, 4)


## Bước 4: Tương quan giữa các đặc trưng

Mục tiêu: phát hiện các cột gần như trùng thông tin (đa cộng tuyến).

Kiến thức: bán kính, chu vi và diện tích liên quan với nhau bằng công thức hình học, nên tương quan gần 1. Đa cộng tuyến làm hệ số của logistic regression khó đọc, nhưng thường không làm giảm khả năng dự đoán.

Kết quả mong đợi: tương quan giữa mean_radius và mean_perimeter khoảng 0,998, giữa mean_radius và mean_area khoảng 0,987.

In [ ]:
# Viết code ở đây:
# 1. Lấy 10 cột bắt đầu bằng 'mean_'
# 2. Vẽ heatmap tương quan (annot=True, fmt='.2f', cmap='coolwarm')
# 3. In tương quan giữa mean_radius với mean_perimeter và mean_area


## Bước 5: Chia train/test và baseline

Mục tiêu: giữ 20% làm tập test và tính accuracy của mô hình đoán mò.

Kiến thức: stratify=y giữ tỷ lệ ác tính giống nhau ở hai tập. DummyClassifier luôn đoán lớp nhiều nhất (B). Mô hình này có recall bằng 0 cho lớp M, tức là bỏ sót mọi ca ung thư.

Kết quả mong đợi: X_train có 455 dòng, X_test có 114 dòng (42 ca ác tính). Baseline accuracy khoảng 0,632, recall bằng 0.

In [ ]:
# Viết code ở đây:
# 1. train_test_split với test_size=0.2, random_state=42, stratify=y
# 2. In kích thước và số ca ác tính trong y_test
# 3. Fit DummyClassifier(strategy='most_frequent'), in accuracy và recall trên tập test


## Bước 6: Logistic regression và confusion matrix

Mục tiêu: huấn luyện logistic regression và đọc 4 ô của confusion matrix.

Kiến thức: confusion matrix nhị phân có 4 ô: TN (lành, đoán lành), FP (lành, đoán ác), FN (ác, đoán lành), TP (ác, đoán ác). Precision = TP / (TP + FP): trong các ca đoán là ác, bao nhiêu ca đúng. Recall = TP / (TP + FN): trong các ca ác thật, bắt được bao nhiêu. F1 là trung bình điều hòa của hai số này.

Kết quả mong đợi: TN = 71, FP = 1, FN = 3, TP = 39. Accuracy khoảng 0,965, precision khoảng 0,975, recall khoảng 0,929, F1 khoảng 0,951.

Câu hỏi: trong chẩn đoán ung thư, FN hay FP nguy hiểm hơn? Vậy nên ưu tiên precision hay recall?

In [ ]:
# Viết code ở đây:
# 1. Tạo pipeline StandardScaler + LogisticRegression(max_iter=5000), fit trên tập train
# 2. Dự đoán tập test, lưu vào pred_lr
# 3. In confusion_matrix, rồi tách tn, fp, fn, tp bằng .ravel()
# 4. In accuracy, precision, recall, F1 (làm tròn 3 chữ số)


## Bước 7: Thay đổi ngưỡng quyết định

Mục tiêu: đổi ngưỡng xác suất để bắt được nhiều ca ác tính hơn.

Kiến thức: predict() dùng ngưỡng 0,5: xác suất ác tính từ 0,5 trở lên thì đoán M. Hạ ngưỡng xuống làm recall tăng nhưng precision thường giảm. Ngưỡng là quyết định của người dùng mô hình, không phải của thuật toán.

Kết quả mong đợi: ngưỡng 0,5: FN = 3, FP = 1. Ngưỡng 0,3: FN = 1, FP = 1, recall tăng lên khoảng 0,976. Ngưỡng 0,1: FN vẫn là 1 nhưng FP tăng lên 5 và precision giảm xuống khoảng 0,891.

In [ ]:
# Viết code ở đây:
# 1. proba_lr = lr.predict_proba(X_test)[:, 1]
# 2. Lặp ngưỡng t trong [0.5, 0.3, 0.1]: pred = (proba_lr >= t).astype(int)
# 3. In ngưỡng, số FN, số FP, precision, recall


## Bước 8: Đường ROC và AUC

Mục tiêu: đánh giá mô hình trên mọi ngưỡng cùng lúc.

Kiến thức: đường ROC vẽ TPR (chính là recall) theo FPR (FP / (FP + TN)) khi ngưỡng chạy từ 1 về 0. Đường càng sát góc trên bên trái thì càng tốt. AUC là diện tích dưới đường ROC: 0,5 là đoán ngẫu nhiên, 1 là hoàn hảo. AUC còn có nghĩa: xác suất mô hình chấm điểm một ca ác tính ngẫu nhiên cao hơn một ca lành tính ngẫu nhiên.

Kết quả mong đợi: AUC khoảng 0,996. Đường ROC gần chạm góc trên bên trái.

In [ ]:
# Viết code ở đây:
# 1. In roc_auc_score(y_test, proba_lr) làm tròn 3 chữ số
# 2. Vẽ RocCurveDisplay.from_predictions(y_test, proba_lr), thêm đường chéo từ (0, 0) đến (1, 1)


## Bước 9: Logistic với phạt L1

Mục tiêu: xây mô hình chỉ dùng một phần nhỏ trong 30 đặc trưng.

Kiến thức: penalty="l1" đưa nhiều hệ số về 0, giống Lasso. C là nghịch đảo của độ mạnh regularization: C nhỏ thì phạt mạnh, giữ ít đặc trưng. solver="liblinear" hỗ trợ L1.

Kết quả mong đợi: giữ lại 7 trên 30 đặc trưng, lớn nhất là worst_radius (khoảng 2,119) và mean_concave_points (khoảng 0,983). Recall khoảng 0,952 và AUC khoảng 0,998, không kém mô hình đủ 30 cột.

Câu hỏi: mô hình ít đặc trưng hơn có lợi gì khi đưa vào bệnh viện dùng thật?

In [ ]:
# Viết code ở đây:
# 1. Tạo pipeline StandardScaler + LogisticRegression(penalty='l1', C=0.1, solver='liblinear'), fit
# 2. Lấy coef_ (gợi ý: l1[-1].coef_[0]), đếm số hệ số khác 0
# 3. In tên các đặc trưng được giữ lại và hệ số (làm tròn 3 chữ số)
# 4. In recall và AUC trên tập test


## Bước 10: SVM

Mục tiêu: huấn luyện SVM với kernel RBF và lấy xác suất để vẽ ROC.

Kiến thức: SVC mặc định chỉ cho nhãn và decision_function (khoảng cách có dấu đến đường biên). decision_function đủ để tính AUC vì AUC chỉ cần thứ tự điểm số. probability=True bật thêm predict_proba nhưng chạy chậm hơn.

Kết quả mong đợi: accuracy khoảng 0,974, precision bằng 1,0, recall khoảng 0,929, AUC khoảng 0,995.

In [ ]:
# Viết code ở đây:
# 1. Tạo pipeline StandardScaler + SVC(kernel='rbf'), fit
# 2. In accuracy, precision, recall
# 3. Tính AUC bằng decision_function(X_test)


## Bước 11: Tìm tham số SVM theo recall

Mục tiêu: dùng GridSearchCV với chỉ số phù hợp với bài toán.

Kiến thức: scoring quyết định GridSearchCV coi tổ hợp nào là tốt nhất. Nếu bài toán ưu tiên không bỏ sót ca bệnh, có thể dùng scoring="recall". Nhưng chỉ dùng recall sẽ chọn mô hình đoán tất cả là M. Một lựa chọn cân bằng hơn là "roc_auc" hoặc "f1".

Kết quả mong đợi: best params là C = 10 và gamma = 'scale', CV F1 khoảng 0,967. Trên tập test, recall khoảng 0,929 và F1 khoảng 0,963.

In [ ]:
# Viết code ở đây:
# 1. cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# 2. param_grid = {'svc__C': [0.1, 1, 10, 100], 'svc__gamma': ['scale', 0.001, 0.01, 0.1]}
# 3. GridSearchCV(pipeline SVM, param_grid, cv=cv, scoring='f1'), fit trên tập train
# 4. In best_params_, best_score_, rồi recall và F1 của best_estimator_ trên tập test


## Bước 12: Gaussian Naive Bayes

Mục tiêu: thử một mô hình xác suất rất đơn giản và nhanh.

Kiến thức: Naive Bayes dùng định lý Bayes và giả định các đặc trưng độc lập với nhau khi biết lớp. GaussianNB giả định mỗi đặc trưng có phân phối chuẩn trong mỗi lớp. Bước 4 cho thấy giả định độc lập bị vi phạm mạnh, nhưng mô hình vẫn thường dự đoán khá tốt. Naive Bayes không cần chuẩn hóa.

Kết quả mong đợi: accuracy khoảng 0,939, precision bằng 1,0, recall chỉ khoảng 0,833 (bỏ sót 7 ca ác tính), AUC khoảng 0,993. Khoảng 95,6% xác suất dự đoán nằm dưới 0,01 hoặc trên 0,99.

Câu hỏi: vì sao xác suất của Naive Bayes thường quá gần 0 hoặc 1? Gợi ý: nghĩ về việc đếm cùng một thông tin nhiều lần (radius, perimeter, area).

In [ ]:
# Viết code ở đây:
# 1. Tạo GaussianNB(), fit trên X_train
# 2. In accuracy, precision, recall, AUC (dùng predict_proba)
# 3. In tỷ lệ các xác suất dự đoán nằm dưới 0,01 hoặc trên 0,99


## Bước 13: So sánh đường ROC

Mục tiêu: vẽ ROC của 3 mô hình trên cùng một hình.

Kiến thức: RocCurveDisplay.from_predictions nhận tham số ax để vẽ nhiều đường lên cùng một trục.

Kết quả mong đợi: 3 đường đều sát góc trên bên trái. Đường của Naive Bayes thấp hơn một chút ở vùng FPR nhỏ.

In [ ]:
# Viết code ở đây:
# 1. fig, ax = plt.subplots()
# 2. Vẽ ROC cho logistic (proba_lr), SVM (score_svm), Naive Bayes (proba_nb) với cùng ax
# 3. Thêm đường chéo


## Bước 14: Đường precision-recall

Mục tiêu: xem đánh đổi giữa precision và recall theo ngưỡng.

Kiến thức: khi lớp dương tính hiếm, đường precision-recall và average precision (AP) cho thông tin rõ hơn ROC. Ở bài này lớp M chiếm 37%, nên hai loại đường đều dùng được.

Kết quả mong đợi: AP của Logistic khoảng 0,994, SVM khoảng 0,993, Naive Bayes khoảng 0,989.

In [ ]:
# Viết code ở đây:
# 1. In average_precision_score cho 3 mô hình
# 2. Vẽ PrecisionRecallDisplay.from_predictions cho 3 mô hình trên cùng một ax


## Bước 15: Bảng so sánh bằng cross-validation

Mục tiêu: so sánh các mô hình trên nhiều lần chia để kết luận chắc hơn.

Kiến thức: cross_validate tính nhiều chỉ số cùng lúc. Kết quả là dict có các khóa dạng test_accuracy, test_recall...

Kết quả mong đợi: SVM RBF tuned có recall (khoảng 0,962) và F1 (khoảng 0,971) cao nhất. Logistic có AUC cao nhất (khoảng 0,995). Naive Bayes thấp nhất với recall khoảng 0,887.

Câu hỏi: nếu bạn phải chọn một mô hình cho bác sĩ dùng, bạn chọn mô hình nào và đặt ngưỡng bao nhiêu? Giải thích bằng 2 câu.

In [ ]:
# Viết code ở đây:
# 1. Tạo dict models gồm: Logistic, Logistic L1 (C=0.1), SVM RBF (tham số tốt nhất), Naive Bayes
# 2. Với mỗi mô hình: cross_validate trên toàn bộ X, y với cv và scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
# 3. Lưu trung bình mỗi chỉ số (làm tròn 3 chữ số) vào bảng và in ra


## Bước 16: Tự thử nghiệm

Không có code mẫu ở bước này. Hãy tự chọn ít nhất hai việc sau:

1. Chạy SVM không có StandardScaler. Accuracy giảm bao nhiêu? Giải thích vì sao mean_area (giá trị hàng trăm) lại gây vấn đề.
2. Tìm ngưỡng nhỏ nhất của logistic regression sao cho recall trên tập test bằng 1. Khi đó precision là bao nhiêu?
3. Chỉ dùng 10 cột worst_ để huấn luyện logistic. So sánh AUC với khi dùng đủ 30 cột.
4. Viết ba câu tiếng Anh tóm tắt bảng ở bước 15 cho một bác sĩ không biết machine learning. Ví dụ: "The logistic model detected about ... of malignant tumours in cross-validation."
